# Company Co-Mention Analysis

This notebook analyzes a network of companies co-mentioned in news articles.  
Nodes are companies, edges are co-mentions, and edge weights count how often two companies appear together.

In [1]:
import pandas as pd
import networkx as nx

from networkx.algorithms.community import louvain_communities
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

In [77]:
co_mentions_df = pd.read_csv("company_comention_edges_all.csv")
companies_df = pd.read_csv("nyse_nasdaq_companies_with_revenue_clenaed_JP.csv")

print("Co-mentions rows:", len(co_mentions_df))
print("Companies rows:", len(companies_df))

print(co_mentions_df.head())
print(companies_df.head())

Co-mentions rows: 3180
Companies rows: 600
      source     target  weight      source_name              target_name
0       Q312      Q3884     119       Apple Inc.                   Amazon
1      Q2283       Q312      88        Microsoft               Apple Inc.
2    Q544847    Q790060      82         Qualcomm                 Broadcom
3  Q60238941      Q7414      75  Fox Corporation  The Walt Disney Company
4   Q1113804  Q60238941      72          Comcast          Fox Corporation
      company company_id                 exchange ticker  \
0       Honda      Q9584  New York Stock Exchange    HMC   
1      Nissan     Q20165                   Nasdaq  NSANY   
2      Itochu    Q717093                   Nasdaq  ITOCY   
3  Sony Group     Q41187  New York Stock Exchange   SONY   
4  NTT DoCoMo    Q853958  New York Stock Exchange    DCM   

                   industry    market_cap            market_cap_date  \
0  industrial manufacturing           NaN                        NaN   
1       

In [78]:
G = nx.Graph()

for _, row in co_mentions_df.iterrows():
    G.add_edge(
        row["source"],
        row["target"],
        weight=row["weight"],
        source_name=row["source_name"],
        target_name=row["target_name"]
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 435
Edges: 3180


In [79]:
for node in G.nodes():
    G.nodes[node]["label"] = node

weighted_degree = dict(G.degree(weight="weight"))
nx.set_node_attributes(G, weighted_degree, "weighted_degree")

## Map industries into broader groups

The original industry labels are very detailed, so I group them into broader categories such as Finance, Energy, Technology, Healthcare, and Retail.

In [80]:
industry_group_terms = {
    "Finance": [
        "financial services",
        "financial service activities, except insurance and pension funding",
        "finance",
        "economics of banking",
        "bank",
        "investment",
        "asset management",
        "insurance",
        "insurance industry",
        "life insurance",
        "vehicle insurance",
        "health insurance",
        "health insurance company",
        "risk management",
        "wire transfer",
        "bitcoin",
        "international standard industrial classification",
    ],

    "Energy": [
        "petroleum industry",
        "energy industry",
        "energy company",
        "industrial gas",
    ],

    "Utilities": [
        "public utility",
        "electricity supply company",
        "electricity generation",
        "electric power industry",
        "water supply",
    ],

    "Technology / Telecom": [
        "software industry",
        "software development",
        "enterprise software",
        "information technology",
        "information technology industry",
        "information technology consulting",
        "information and communications technology",
        "computer security",
        "information security",
        "computer and network surveillance",
        "computer industry",
        "computer hardware industry",
        "computer network",
        "computer storage media",
        "computer-aided design",
        "networking hardware",
        "internet",
        "web hosting service",
        "technology",
        "technology company",
        "artificial intelligence",
        "analytics",
        "automation",
        "robotics",
        "3d printing",
        "semiconductor industry",
        "electronics",
        "consumer electronics industry",
        "telecommunications",
        "communication",
        "video conference",
        "telepresence",
        "digital distribution",
        "electrical industry",
    ],

    "Healthcare": [
        "pharmaceutical industry",
        "biotechnology",
        "biotechnology industry",
        "health care",
        "health technology",
        "medical technology industry",
        "medical equipment",
        "managed care",
        "life sciences",
        "high-performance liquid chromatography",
    ],

    "Automotive": [
        "automotive industry",
        "car rental company",
    ],

    "Industrials / Aerospace / Defense": [
        "industrial manufacturing",
        "industrial sector",
        "mechanical engineering",
        "engineering",
        "aerospace industry",
        "aerospace engineering",
        "aviation",
        "weapons industry",
        "defense contractor",
        "manufacture of machinery and equipment",
        "equipment rental",
        "power tool",
        "shipbuilding",
        "construction",
        "facility management",
        "outsourcing",
    ],

    "Materials / Mining / Chemicals": [
        "chemical industry",
        "pesticide and other agricultural chemical manufacturing",
        "iron and steel industry",
        "mining",
        "mining industry",
        "metal",
        "cement industry",
        "building materials trade",
        "glass",
        "pulp and paper industry",
    ],

    "Consumer / Retail / Food": [
        "retail",
        "wholesale",
        "trade",
        "e-commerce",
        "direct selling",
        "auction",
        "product distribution",
        "final good",
        "hardware store",
        "fast-moving consumer goods",
        "personal care product",
        "cosmetics industry",
        "food industry",
        "food processing",
        "food service",
        "restaurant",
        "fast food",
        "fast casual restaurant",
        "system catering",
        "beverage industry",
        "brewing industry",
        "coffee industry",
        "manufacture of cocoa, chocolate and sugar confectionery",
        "alcohol industry",
        "tobacco industry",
        "clothing industry",
        "footwear industry",
        "textile industry",
        "cannabis industry",
    ],

    "Media / Entertainment": [
        "media industry",
        "mass media",
        "show business",
        "streaming media",
        "broadcasting",
        "broadcast television system",
        "television",
        "terrestrial television",
        "radio broadcasting",
        "journalism",
        "music industry",
        "production music",
        "animation",
        "video game industry",
        "game industry",
        "sports industry",
        "gambling",
        "gambling industry",
    ],

    "Transport / Logistics": [
        "logistics",
        "transport",
        "transport industry",
        "freight transport industry",
        "air transport",
        "rail transport",
        "water transport",
        "shipping line",
        "waste management",
        "waste management industry",
    ],

    "Real Estate": [
        "real estate industry",
        "real estate investment trust",
        "self storage",
    ],

    "Travel / Hospitality": [
        "tourism",
        "tourism industry",
        "hospitality industry",
        "space tourism",
    ],

    "Agriculture": [
        "agriculture",
        "agribusiness",
    ],

    "Professional Services": [
        "professional service",
        "consulting company",
        "marketing",
        "e-recruitment",
    ],

    "Holding / Conglomerate": [
        "holding company",
        "holding company activities",
        "conglomerate",
    ],

    "Other / Unclear": [
        "tertiary sector of the economy",
        "quaternary sector of the economy",
    ],
}

In [81]:
industry_group_terms_lower = {
    group: {term.lower().strip() for term in terms}
    for group, terms in industry_group_terms.items()
}

def map_to_industry_group(industry):
    if pd.isna(industry):
        return "Unknown"

    industry = str(industry).lower().strip()

    for group, terms in industry_group_terms_lower.items():
        if industry in terms:
            return group

    return "Other / Unmapped"

companies_df["industry_group"] = companies_df["industry"].apply(map_to_industry_group)

companies_df[["company", "industry", "industry_group"]].head(20)

,company,industry,industry_group
0,Honda,industrial manufacturing,Industrials / Aerospace / Defense
1,Nissan,automotive industry,Automotive
2,Itochu,trade,Consumer / Retail / Food
3,Sony Group,video game industry,Media / Entertainment
4,NTT DoCoMo,telecommunications,Technology / Telecom
5,Canon Inc.,electronics,Technology / Telecom
6,Micro Focus International,information technology,Technology / Telecom
7,Sinopec,petroleum industry,Energy
8,PetroChina Company Limited,petroleum industry,Energy
9,TSMC,semiconductor industry,Technology / Telecom


In [82]:
companies_df["industry_group"].value_counts()

industry_group
Technology / Telecom                 115
Consumer / Retail / Food              75
Finance                               75
Industrials / Aerospace / Defense     54
Unknown                               53
Energy                                49
Healthcare                            44
Media / Entertainment                 29
Automotive                            25
Materials / Mining / Chemicals        23
Transport / Logistics                 17
Utilities                             13
Travel / Hospitality                   8
Real Estate                            6
Holding / Conglomerate                 5
Professional Services                  4
Agriculture                            3
Other / Unclear                        2
Name: count, dtype: int64

##  Add company metadata to graph nodes

The graph nodes are company IDs, so I use `company_id` to add company names, industries, and industry groups to each node.

In [83]:
# Make sure IDs are clean strings
companies_df["company_id"] = companies_df["company_id"].astype(str).str.strip()

# Your graph nodes are company IDs, so map by company_id
company_name_map = dict(
    zip(companies_df["company_id"], companies_df["company"])
)

industry_map = dict(
    zip(companies_df["company_id"], companies_df["industry"])
)

industry_group_map = dict(
    zip(companies_df["company_id"], companies_df["industry_group"])
)

for node in G.nodes():
    node_id = str(node).strip()

    G.nodes[node]["label"] = company_name_map.get(node_id, node_id)
    G.nodes[node]["industry"] = industry_map.get(node_id, "Unknown")
    G.nodes[node]["industry_group"] = industry_group_map.get(node_id, "Unknown")

In [84]:
node_industry_check = pd.DataFrame([
    {
        "company_id": node,
        "company": G.nodes[node].get("label", node),
        "industry": G.nodes[node].get("industry", "Unknown"),
        "industry_group": G.nodes[node].get("industry_group", "Unknown"),
    }
    for node in G.nodes()
])

node_industry_check.head(20)

,company_id,company,industry,industry_group
0,Q312,Apple Inc.,digital distribution,Technology / Telecom
1,Q3884,Amazon,retail,Consumer / Retail / Food
2,Q2283,Microsoft,software development,Technology / Telecom
3,Q544847,Qualcomm,telecommunications,Technology / Telecom
4,Q790060,Broadcom,semiconductor industry,Technology / Telecom
5,Q60238941,Fox Corporation,media industry,Media / Entertainment
6,Q7414,The Walt Disney Company,animation,Media / Entertainment
7,Q1113804,Comcast,telecommunications,Technology / Telecom
8,Q193326,Goldman Sachs,International Standard Industrial Classification,Finance
9,Q334204,Morgan Stanley,financial services,Finance


##  Louvain community detection

I use Louvain to find communities of companies that are strongly connected through repeated co-mentions.

In [85]:
communities = louvain_communities(
    G,
    weight="weight",
    resolution=1,
    seed=42
)

for cluster_id, community in enumerate(communities):
    for company in community:
        G.nodes[company]["cluster"] = cluster_id

print("Found clusters:", len(communities))

Found clusters: 11


## Create node table

I convert graph nodes into a dataframe containing company name, Louvain cluster, industry group, and weighted degree.

In [113]:
node_df = pd.DataFrame([
    {
        "company_id": node,
        "company": G.nodes[node].get("label", node),
        "cluster": G.nodes[node].get("cluster"),
        "industry": G.nodes[node].get("industry", "Unknown"),
        "industry_group": G.nodes[node].get("industry_group", "Unknown"),
        "weighted_degree": G.nodes[node].get("weighted_degree", 0),
    }
    for node in G.nodes()
])

In [114]:
eval_df = node_df[
    ~node_df["industry_group"].isin(["Unknown", "Other / Unmapped"])
].copy()

print("Total graph nodes:", len(node_df))
print("Nodes used for evaluation:", len(eval_df))

eval_df.head()

Total graph nodes: 435
Nodes used for evaluation: 407


,company_id,company,cluster,industry,industry_group,weighted_degree
0,Q312,Apple Inc.,0,digital distribution,Technology / Telecom,836
1,Q3884,Amazon,0,retail,Consumer / Retail / Food,689
2,Q2283,Microsoft,0,software development,Technology / Telecom,570
3,Q544847,Qualcomm,4,telecommunications,Technology / Telecom,263
4,Q790060,Broadcom,4,semiconductor industry,Technology / Telecom,208


## Compare Louvain clusters with industry groups

I compare the detected graph communities with the known industry groups using a contingency table, purity, NMI, and ARI.

In [115]:
contingency = pd.crosstab(
    eval_df["cluster"],
    eval_df["industry_group"]
)

contingency

industry_group,Agriculture,Automotive,Consumer / Retail / Food,Energy,Finance,Healthcare,Holding / Conglomerate,Industrials / Aerospace / Defense,Materials / Mining / Chemicals,Media / Entertainment,Other / Unclear,Professional Services,Real Estate,Technology / Telecom,Transport / Logistics,Travel / Hospitality,Utilities
cluster,,,,,,,,,,,,,,,,,
0,0,0,19,2,8,3,1,5,0,5,0,2,0,31,3,2,0
1,0,0,2,1,0,1,0,0,1,8,0,0,1,14,0,1,0
2,0,1,1,6,12,1,3,17,3,0,0,0,0,7,2,0,1
3,0,3,1,2,33,0,0,0,2,2,1,0,3,5,0,0,1
4,0,4,6,1,4,2,0,5,0,0,0,0,0,20,1,1,2
5,0,9,1,1,3,0,0,1,0,1,0,0,1,5,1,0,3
6,0,1,1,0,1,23,0,2,0,0,0,0,0,4,0,1,1
7,1,0,0,23,1,0,0,3,1,0,0,0,0,1,0,0,2
8,1,0,22,2,1,3,0,2,7,2,0,0,0,6,2,1,0


In [116]:
purity = contingency.max(axis=1).sum() / contingency.values.sum()

print("Purity:", purity)

Purity: 0.47911547911547914


In [90]:
cluster_summary_rows = []

for cluster_id, group in eval_df.groupby("cluster"):
    industry_counts = group["industry_group"].value_counts()

    dominant_industry = industry_counts.index[0]
    dominant_count = industry_counts.iloc[0]
    cluster_size = len(group)

    cluster_summary_rows.append({
        "cluster": cluster_id,
        "cluster_size": cluster_size,
        "dominant_industry": dominant_industry,
        "dominant_count": dominant_count,
        "cluster_purity": dominant_count / cluster_size,
        "all_industries": dict(industry_counts),
    })

cluster_summary_df = pd.DataFrame(cluster_summary_rows)

cluster_summary_df = cluster_summary_df.sort_values(
    "cluster_size",
    ascending=False
)

cluster_summary_df

,cluster,cluster_size,dominant_industry,dominant_count,cluster_purity,all_industries
0,0,81,Technology / Telecom,31,0.382716,"{'Technology / Telecom': 31, 'Consumer / Retai..."
2,2,54,Industrials / Aerospace / Defense,17,0.314815,"{'Industrials / Aerospace / Defense': 17, 'Fin..."
3,3,53,Finance,33,0.622642,"{'Finance': 33, 'Technology / Telecom': 5, 'Au..."
8,8,49,Consumer / Retail / Food,22,0.448980,"{'Consumer / Retail / Food': 22, 'Materials / ..."
4,4,46,Technology / Telecom,20,0.434783,"{'Technology / Telecom': 20, 'Consumer / Retai..."
6,6,34,Healthcare,23,0.676471,"{'Healthcare': 23, 'Technology / Telecom': 4, ..."
7,7,32,Energy,23,0.718750,"{'Energy': 23, 'Industrials / Aerospace / Defe..."
1,1,29,Technology / Telecom,14,0.482759,"{'Technology / Telecom': 14, 'Media / Entertai..."
5,5,26,Automotive,9,0.346154,"{'Automotive': 9, 'Technology / Telecom': 5, '..."
9,9,2,Transport / Logistics,2,1.000000,{'Transport / Logistics': 2}


In [91]:
y_true_industry = eval_df["industry_group"]
y_pred_cluster = eval_df["cluster"]

nmi = normalized_mutual_info_score(y_true_industry, y_pred_cluster)
ari = adjusted_rand_score(y_true_industry, y_pred_cluster)

print("Purity:", purity)
print("NMI:", nmi)
print("ARI:", ari)

Purity: 0.47911547911547914
NMI: 0.3053317482096056
ARI: 0.1659498428070293


## Visualize the network

I visualize the same graph twice: once colored by Louvain cluster and once colored by industry group.

In [117]:
from ipysigma import Sigma

Sigma(
    G,
    node_label="label",          # company name
    node_color="cluster",        # Louvain cluster
    node_size="weighted_degree", # more connected companies are bigger
    edge_size="weight",          # stronger co-mentions are thicker
    default_edge_type="curve",
    clickable_edges=False
)

Sigma(nx.Graph with 435 nodes and 3,180 edges)

In [118]:
Sigma(
    G,
    node_label="label",
    node_color="industry_group", # industry categories
    node_size="weighted_degree",
    edge_size="weight",
    default_edge_type="curve",
    clickable_edges=False
)

Sigma(nx.Graph with 435 nodes and 3,180 edges)

## Robustness test with edge thresholds

I test whether removing weak co-mentions changes the agreement between Louvain clusters and industry groups.

In [119]:
results = []

for min_weight in [1, 2, 3, 5, 10]:
    filtered_edges = co_mentions_df[
        co_mentions_df["weight"] >= min_weight
    ].copy()

    G_temp = nx.Graph()

    for _, row in filtered_edges.iterrows():
        G_temp.add_edge(
            row["source"],
            row["target"],
            weight=row["weight"]
        )

    # skip empty graphs
    if G_temp.number_of_nodes() == 0 or G_temp.number_of_edges() == 0:
        continue

    # add node attributes
    for node in G_temp.nodes():
        node_id = str(node).strip()

        G_temp.nodes[node]["label"] = company_name_map.get(node_id, node_id)
        G_temp.nodes[node]["industry"] = industry_map.get(node_id, "Unknown")
        G_temp.nodes[node]["industry_group"] = industry_group_map.get(node_id, "Unknown")

    weighted_degree_temp = dict(G_temp.degree(weight="weight"))
    nx.set_node_attributes(G_temp, weighted_degree_temp, "weighted_degree")

    # Louvain
    communities_temp = louvain_communities(
        G_temp,
        weight="weight",
        resolution=1,
        seed=42
    )

    for cluster_id, community in enumerate(communities_temp):
        for node in community:
            G_temp.nodes[node]["cluster"] = cluster_id

    # node dataframe
    node_df_temp = pd.DataFrame([
        {
            "company_id": node,
            "company": G_temp.nodes[node].get("label", node),
            "cluster": G_temp.nodes[node].get("cluster"),
            "industry_group": G_temp.nodes[node].get("industry_group", "Unknown"),
            "weighted_degree": G_temp.nodes[node].get("weighted_degree", 0),
        }
        for node in G_temp.nodes()
    ])

    eval_temp = node_df_temp[
        ~node_df_temp["industry_group"].isin(["Unknown", "Other / Unmapped"])
    ].copy()

    if len(eval_temp) == 0:
        continue

    contingency_temp = pd.crosstab(
        eval_temp["cluster"],
        eval_temp["industry_group"]
    )

    purity_temp = (
        contingency_temp.max(axis=1).sum()
        / contingency_temp.values.sum()
    )

    nmi_temp = normalized_mutual_info_score(
        eval_temp["industry_group"],
        eval_temp["cluster"]
    )

    ari_temp = adjusted_rand_score(
        eval_temp["industry_group"],
        eval_temp["cluster"]
    )

    results.append({
        "min_edge_weight": min_weight,
        "nodes": G_temp.number_of_nodes(),
        "edges": G_temp.number_of_edges(),
        "evaluated_nodes": len(eval_temp),
        "clusters": len(communities_temp),
        "purity": purity_temp,
        "nmi": nmi_temp,
        "ari": ari_temp,
    })
   

threshold_results_df = pd.DataFrame(results)

threshold_results_df

,min_edge_weight,nodes,edges,evaluated_nodes,clusters,purity,nmi,ari
0,1,435,3180,407,11,0.479115,0.305332,0.165950
1,2,281,1338,263,15,0.532319,0.393674,0.208574
2,3,228,790,215,13,0.516279,0.413917,0.216936
3,5,157,376,151,19,0.629139,0.546117,0.265946
4,10,94,158,93,19,0.763441,0.663662,0.354476


## Node centrality

I compute centrality measures to identify the most important companies in the co-mention network.

In [120]:
# Degree centrality: number of distinct neighbors, normalized
degree_centrality = nx.degree_centrality(G)

# Weighted degree: sum of edge weights
weighted_degree = dict(G.degree(weight="weight"))

for u, v, data in G.edges(data=True):
    data["distance"] = 1 / data["weight"]

betweenness = nx.betweenness_centrality(
    G,
    weight="distance",
    normalized=True
)
# PageRank: overall network importance
pagerank = nx.pagerank(
    G,
    weight="weight"
)

In [121]:
centrality_df = pd.DataFrame([
    {
        "company_id": node,
        "company": G.nodes[node].get("label", node),
        "industry_group": G.nodes[node].get("industry_group", "Unknown"),
        "degree": G.degree(node),
        "degree_centrality": degree_centrality[node],
        "weighted_degree": weighted_degree[node],
        "betweenness": betweenness[node],
        "pagerank": pagerank[node],
    }
    for node in G.nodes()
])

centrality_df.head()

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank
0,Q312,Apple Inc.,Technology / Telecom,135,0.311060,836,0.396108,0.034175
1,Q3884,Amazon,Consumer / Retail / Food,108,0.248848,689,0.290968,0.027882
2,Q2283,Microsoft,Technology / Telecom,103,0.237327,570,0.111621,0.023369
3,Q544847,Qualcomm,Technology / Telecom,59,0.135945,263,0.018550,0.010329
4,Q790060,Broadcom,Technology / Telecom,42,0.096774,208,0.000000,0.008062


## Most central companies

I rank companies by weighted degree, betweenness, and PageRank.

In [122]:
centrality_df.sort_values(
    "weighted_degree",
    ascending=False
).head(20)

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank
0,Q312,Apple Inc.,Technology / Telecom,135,0.311060,836,0.396108,0.034175
1,Q3884,Amazon,Consumer / Retail / Food,108,0.248848,689,0.290968,0.027882
24,Q1472929,"Nasdaq, Inc.",Finance,183,0.421659,600,0.378384,0.035326
9,Q334204,Morgan Stanley,Finance,116,0.267281,576,0.258054,0.023773
2,Q2283,Microsoft,Technology / Telecom,103,0.237327,570,0.111621,0.023369
8,Q193326,Goldman Sachs,Finance,107,0.246544,498,0.086206,0.021103
12,Q20800404,Alphabet Inc.,Technology / Telecom,71,0.163594,442,0.091153,0.016846
13,Q66048,Deutsche Bank,Finance,83,0.191244,381,0.052527,0.015404
10,Q219508,Citigroup,Finance,80,0.184332,370,0.062696,0.015476
16,Q81965,General Motors,Automotive,69,0.158986,364,0.067202,0.013571


In [123]:
centrality_df.sort_values(
    "betweenness",
    ascending=False
).head(20)

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank
0,Q312,Apple Inc.,Technology / Telecom,135,0.311060,836,0.396108,0.034175
24,Q1472929,"Nasdaq, Inc.",Finance,183,0.421659,600,0.378384,0.035326
1,Q3884,Amazon,Consumer / Retail / Food,108,0.248848,689,0.290968,0.027882
9,Q334204,Morgan Stanley,Finance,116,0.267281,576,0.258054,0.023773
26,Q66,Boeing,Industrials / Aerospace / Defense,91,0.209677,316,0.156772,0.016016
61,Q157062,Unilever,Consumer / Retail / Food,52,0.119816,183,0.114008,0.008239
2,Q2283,Microsoft,Technology / Telecom,103,0.237327,570,0.111621,0.023369
17,Q35476,AT&T,Technology / Telecom,84,0.193548,349,0.095533,0.014067
12,Q20800404,Alphabet Inc.,Technology / Telecom,71,0.163594,442,0.091153,0.016846
8,Q193326,Goldman Sachs,Finance,107,0.246544,498,0.086206,0.021103


In [124]:
centrality_df.sort_values(
    "pagerank",
    ascending=False
).head(20)

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank
24,Q1472929,"Nasdaq, Inc.",Finance,183,0.421659,600,0.378384,0.035326
0,Q312,Apple Inc.,Technology / Telecom,135,0.311060,836,0.396108,0.034175
1,Q3884,Amazon,Consumer / Retail / Food,108,0.248848,689,0.290968,0.027882
9,Q334204,Morgan Stanley,Finance,116,0.267281,576,0.258054,0.023773
2,Q2283,Microsoft,Technology / Telecom,103,0.237327,570,0.111621,0.023369
8,Q193326,Goldman Sachs,Finance,107,0.246544,498,0.086206,0.021103
12,Q20800404,Alphabet Inc.,Technology / Telecom,71,0.163594,442,0.091153,0.016846
26,Q66,Boeing,Industrials / Aerospace / Defense,91,0.209677,316,0.156772,0.016016
10,Q219508,Citigroup,Finance,80,0.184332,370,0.062696,0.015476
13,Q66048,Deutsche Bank,Finance,83,0.191244,381,0.052527,0.015404


## Add revenue and market capitalization

I merge centrality results with company revenue and market capitalization to test whether larger companies are more central.

In [125]:
analysis_df = centrality_df.merge(
    companies_df[
        [
            "company_id",
            "company",
            "industry",
            "industry_group",
            "revenue",
            "market_cap"
        ]
    ],
    on="company_id",
    how="left",
    suffixes=("", "_meta")
)

analysis_df.head()

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank,company_meta,industry,industry_group_meta,revenue,market_cap
0,Q312,Apple Inc.,Technology / Telecom,135,0.311060,836,0.396108,0.034175,Apple Inc.,digital distribution,Technology / Telecom,4.161610e+11,3.205000e+12
1,Q3884,Amazon,Consumer / Retail / Food,108,0.248848,689,0.290968,0.027882,Amazon,retail,Consumer / Retail / Food,6.379590e+11,2.018000e+12
2,Q2283,Microsoft,Technology / Telecom,103,0.237327,570,0.111621,0.023369,Microsoft,software development,Technology / Telecom,2.817240e+11,3.162000e+12
3,Q544847,Qualcomm,Technology / Telecom,59,0.135945,263,0.018550,0.010329,Qualcomm,telecommunications,Technology / Telecom,4.428400e+10,1.889300e+11
4,Q790060,Broadcom,Technology / Telecom,42,0.096774,208,0.000000,0.008062,Broadcom,semiconductor industry,Technology / Telecom,3.581900e+10,NaN


In [106]:
analysis_df["revenue"] = pd.to_numeric(analysis_df["revenue"], errors="coerce")
analysis_df["market_cap"] = pd.to_numeric(analysis_df["market_cap"], errors="coerce")

analysis_df[["company", "revenue", "market_cap", "weighted_degree", "pagerank", "betweenness"]].head()

,company,revenue,market_cap,weighted_degree,pagerank,betweenness
0,Apple Inc.,4.161610e+11,3.205000e+12,836,0.034175,0.063224
1,Amazon,6.379590e+11,2.018000e+12,689,0.027882,0.038479
2,Microsoft,2.817240e+11,3.162000e+12,570,0.023369,0.037876
3,Qualcomm,4.428400e+10,1.889300e+11,263,0.010329,0.015232
4,Broadcom,3.581900e+10,NaN,208,0.008062,0.009565


## Company size and centrality

I use Pearson and Spearman correlations to test whether revenue or market capitalization is associated with centrality.

In [107]:
import numpy as np

analysis_df["log_revenue"] = np.log1p(analysis_df["revenue"])
analysis_df["log_market_cap"] = np.log1p(analysis_df["market_cap"])
analysis_df["log_weighted_degree"] = np.log1p(analysis_df["weighted_degree"])
analysis_df["log_degree"] = np.log1p(analysis_df["degree"])
analysis_df["log_pagerank"] = np.log1p(analysis_df["pagerank"])
analysis_df["log_betweenness"] = np.log1p(analysis_df["betweenness"])

## Centrality by company size quartiles

I divide companies into revenue and market-cap quartiles and compare median centrality across groups.

In [139]:
analysis_df["revenue_quartile"] = pd.qcut(
    analysis_df["revenue"],
    q=4,
    labels=["Q1 lowest revenue", "Q2", "Q3", "Q4 highest revenue"]
)

revenue_quartile_summary = analysis_df.groupby("revenue_quartile").agg(
    n_companies=("company_id", "count"),
    median_degree=("degree", "median"),
    median_weighted_degree=("weighted_degree", "median"),
    median_pagerank=("pagerank", "median"),
    median_betweenness=("betweenness", "median")
).reset_index()

revenue_quartile_summary

,revenue_quartile,n_companies,median_degree,median_weighted_degree,median_pagerank,median_betweenness
0,Q1 lowest revenue,109,3.0,3.0,0.000545,0.000000
1,Q2,109,5.0,6.0,0.000778,0.000000
2,Q3,108,12.5,25.0,0.001671,0.000000
3,Q4 highest revenue,109,18.0,45.0,0.002424,0.001485


In [140]:
analysis_df["market_cap_quartile"] = pd.qcut(
    analysis_df["market_cap"],
    q=4,
    labels=["Q1 lowest market cap", "Q2", "Q3", "Q4 highest market cap"]
)

marketcap_quartile_summary = analysis_df.groupby("market_cap_quartile").agg(
    n_companies=("company_id", "count"),
    median_degree=("degree", "median"),
    median_weighted_degree=("weighted_degree", "median"),
    median_pagerank=("pagerank", "median"),
    median_betweenness=("betweenness", "median")
).reset_index()

marketcap_quartile_summary

,market_cap_quartile,n_companies,median_degree,median_weighted_degree,median_pagerank,median_betweenness
0,Q1 lowest market cap,30,8.5,13.5,0.000953,0.000000
1,Q2,29,14.0,29.0,0.001877,0.000282
2,Q3,29,31.0,64.0,0.003338,0.006167
3,Q4 highest market cap,30,36.0,86.5,0.004045,0.011550


In [141]:
from scipy.stats import kruskal

# Revenue quartiles
groups = [
    group["weighted_degree"].dropna().values
    for _, group in analysis_df.groupby("revenue_quartile", observed=True)
]

stat, p = kruskal(*groups)

print("Revenue quartiles - weighted degree")
print("Kruskal-Wallis statistic:", stat)
print("p-value:", p)

Revenue quartiles - weighted degree
Kruskal-Wallis statistic: 134.7346151171232
p-value: 5.159622322782995e-29


In [142]:
# Market-cap quartiles
groups = [
    group["weighted_degree"].dropna().values
    for _, group in analysis_df.groupby("market_cap_quartile", observed=True)
]

stat, p = kruskal(*groups)

print("Market-cap quartiles - weighted degree")
print("Kruskal-Wallis statistic:", stat)
print("p-value:", p)

Market-cap quartiles - weighted degree
Kruskal-Wallis statistic: 26.709473860856225
p-value: 6.773544910252422e-06


## Centrality by industry

I compare centrality across industry groups using descriptive statistics and the Kruskal-Wallis test.

In [111]:
from scipy.stats import kruskal

metric = "weighted_degree"

groups = [
    group[metric].dropna().values
    for _, group in analysis_df.groupby("industry_group")
    if len(group) >= 5
]

stat, p = kruskal(*groups)

print("Metric:", metric)
print("Kruskal-Wallis statistic:", stat)
print("p-value:", p)

Metric: weighted_degree
Kruskal-Wallis statistic: 27.654017562466226
p-value: 0.010109889992686975


In [112]:
industry_centrality = analysis_df.groupby("industry_group").agg(
    n_companies=("company_id", "count"),
    median_weighted_degree=("weighted_degree", "median"),
    mean_weighted_degree=("weighted_degree", "mean"),
    q1_weighted_degree=("weighted_degree", lambda x: x.quantile(0.25)),
    q3_weighted_degree=("weighted_degree", lambda x: x.quantile(0.75)),
    median_pagerank=("pagerank", "median"),
    median_betweenness=("betweenness", "median"),
).reset_index()

industry_centrality["iqr_weighted_degree"] = (
    industry_centrality["q3_weighted_degree"] 
    - industry_centrality["q1_weighted_degree"]
)

industry_centrality.sort_values("median_weighted_degree", ascending=False)

,industry_group,n_companies,median_weighted_degree,mean_weighted_degree,q1_weighted_degree,q3_weighted_degree,median_pagerank,median_betweenness,iqr_weighted_degree
14,Transport / Logistics,11,31.0,27.636364,3.00,43.50,0.001886,0.000160,40.50
1,Automotive,18,25.0,83.555556,5.25,126.50,0.001341,0.001530,121.25
2,Consumer / Retail / Food,53,21.0,50.132075,5.00,53.00,0.001350,0.001292,48.00
4,Finance,63,19.0,71.158730,3.50,55.00,0.001187,0.001531,51.50
0,Agriculture,2,17.5,17.500000,11.25,23.75,0.001200,0.000594,12.50
5,Healthcare,33,17.0,29.878788,3.00,51.00,0.001222,0.001120,48.00
9,Media / Entertainment,18,17.0,56.888889,3.75,52.25,0.000959,0.000250,48.50
6,Holding / Conglomerate,4,16.5,17.250000,2.50,31.25,0.001050,0.000233,28.75
8,Materials / Mining / Chemicals,14,13.0,16.000000,6.50,26.50,0.001000,0.000967,20.00
13,Technology / Telecom,93,10.0,57.741935,3.00,48.00,0.000905,0.000814,45.00


# Which companies are unusually central?

In [130]:
def iqr_outlier_table(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    threshold = q3 + 1.5 * iqr

    outliers = df[df[column] > threshold].copy()
    outliers = outliers.sort_values(column, ascending=False)

    return outliers, threshold

weighted_degree_outliers, wd_outlier_threshold = iqr_outlier_table(
    centrality_df,
    "weighted_degree"
)

print("Weighted degree outlier threshold:", wd_outlier_threshold)

weighted_degree_outliers[
    ["company", "industry_group", "degree", "weighted_degree", "pagerank", "betweenness"]
].head(30)


Weighted degree outlier threshold: 85.5


,company,industry_group,degree,weighted_degree,pagerank,betweenness
0,Apple Inc.,Technology / Telecom,135,836,0.034175,0.396108
1,Amazon,Consumer / Retail / Food,108,689,0.027882,0.290968
24,"Nasdaq, Inc.",Finance,183,600,0.035326,0.378384
9,Morgan Stanley,Finance,116,576,0.023773,0.258054
2,Microsoft,Technology / Telecom,103,570,0.023369,0.111621
8,Goldman Sachs,Finance,107,498,0.021103,0.086206
12,Alphabet Inc.,Technology / Telecom,71,442,0.016846,0.091153
13,Deutsche Bank,Finance,83,381,0.015404,0.052527
10,Citigroup,Finance,80,370,0.015476,0.062696
16,General Motors,Automotive,69,364,0.013571,0.067202


In [131]:
weighted_degree_outliers["industry_group"].value_counts()

industry_group
Technology / Telecom                 14
Finance                              11
Automotive                            8
Consumer / Retail / Food              6
Industrials / Aerospace / Defense     4
Media / Entertainment                 3
Healthcare                            2
Energy                                2
Name: count, dtype: int64

# Are popular companies also bridge companies

In [132]:

temp = centrality_df[["weighted_degree", "betweenness"]].dropna()

spearman_corr, spearman_p = spearmanr(
    temp["weighted_degree"],
    temp["betweenness"]
)

pearson_corr, pearson_p = pearsonr(
    temp["weighted_degree"],
    temp["betweenness"]
)

print("Spearman correlation:", spearman_corr)
print("Spearman p-value:", spearman_p)

print("Pearson correlation:", pearson_corr)
print("Pearson p-value:", pearson_p)

Spearman correlation: 0.7090842706193246
Spearman p-value: 1.0731443882894945e-67
Pearson correlation: 0.8340010604544227
Pearson p-value: 6.928717731210973e-114


In [133]:
top_n = 20

top_weighted = centrality_df.sort_values(
    "weighted_degree",
    ascending=False
).head(top_n)

top_betweenness = centrality_df.sort_values(
    "betweenness",
    ascending=False
).head(top_n)

top_weighted_set = set(top_weighted["company_id"])
top_betweenness_set = set(top_betweenness["company_id"])

overlap = top_weighted_set & top_betweenness_set

print("Top weighted-degree companies:", len(top_weighted_set))
print("Top betweenness companies:", len(top_betweenness_set))
print("Overlap:", len(overlap))
print("Overlap share:", len(overlap) / top_n)

Top weighted-degree companies: 20
Top betweenness companies: 20
Overlap: 13
Overlap share: 0.65


In [134]:
centrality_df[
    centrality_df["company_id"].isin(overlap)
][
    ["company", "industry_group", "weighted_degree", "betweenness", "pagerank"]
].sort_values("betweenness", ascending=False)

,company,industry_group,weighted_degree,betweenness,pagerank
0,Apple Inc.,Technology / Telecom,836,0.396108,0.034175
24,"Nasdaq, Inc.",Finance,600,0.378384,0.035326
1,Amazon,Consumer / Retail / Food,689,0.290968,0.027882
9,Morgan Stanley,Finance,576,0.258054,0.023773
26,Boeing,Industrials / Aerospace / Defense,316,0.156772,0.016016
2,Microsoft,Technology / Telecom,570,0.111621,0.023369
17,AT&T,Technology / Telecom,349,0.095533,0.014067
12,Alphabet Inc.,Technology / Telecom,442,0.091153,0.016846
8,Goldman Sachs,Finance,498,0.086206,0.021103
16,General Motors,Automotive,364,0.067202,0.013571


In [135]:
popular_not_bridge = top_weighted[
    ~top_weighted["company_id"].isin(top_betweenness_set)
]

popular_not_bridge[
    ["company", "industry_group", "weighted_degree", "betweenness", "pagerank"]
]

,company,industry_group,weighted_degree,betweenness,pagerank
18,Intel,Technology / Telecom,312,0.023063,0.012754
11,Walmart,Consumer / Retail / Food,290,0.023483,0.012475
14,UBS,Finance,278,0.024418,0.011344
5,Fox Corporation,Media / Entertainment,269,0.013357,0.009529
3,Qualcomm,Technology / Telecom,263,0.018550,0.010329
6,The Walt Disney Company,Media / Entertainment,261,0.006705,0.009210
30,Tesla,Automotive,246,0.020274,0.009715


In [136]:
bridge_not_popular = top_betweenness[
    ~top_betweenness["company_id"].isin(top_weighted_set)
]

bridge_not_popular[
    ["company", "industry_group", "weighted_degree", "betweenness", "pagerank"]
]

,company,industry_group,weighted_degree,betweenness,pagerank
61,Unilever,Consumer / Retail / Food,183,0.114008,0.008239
31,Shell,Energy,99,0.065453,0.005881
32,Chevron Corporation,Energy,90,0.044369,0.006475
20,IBM,Technology / Telecom,201,0.040992,0.009801
28,General Electric,Industrials / Aerospace / Defense,199,0.034147,0.010606
121,The Home Depot,Consumer / Retail / Food,70,0.034051,0.003838
36,Target Corporation,Consumer / Retail / Food,140,0.033672,0.006474


# Are centrality outliers concentrated in certain industries?

In [137]:
outlier_industry_counts = weighted_degree_outliers["industry_group"].value_counts()
all_industry_counts = centrality_df["industry_group"].value_counts()

outlier_industry_comparison = pd.DataFrame({
    "all_companies": all_industry_counts,
    "centrality_outliers": outlier_industry_counts
}).fillna(0)

outlier_industry_comparison["all_company_share"] = (
    outlier_industry_comparison["all_companies"]
    / outlier_industry_comparison["all_companies"].sum()
)

outlier_industry_comparison["outlier_share"] = (
    outlier_industry_comparison["centrality_outliers"]
    / outlier_industry_comparison["centrality_outliers"].sum()
)

outlier_industry_comparison["overrepresentation"] = (
    outlier_industry_comparison["outlier_share"]
    / outlier_industry_comparison["all_company_share"]
)

outlier_industry_comparison = outlier_industry_comparison.sort_values(
    "centrality_outliers",
    ascending=False
)

outlier_industry_comparison

,all_companies,centrality_outliers,all_company_share,outlier_share,overrepresentation
industry_group,,,,,
Technology / Telecom,93,14.0,0.213793,0.28,1.309677
Finance,63,11.0,0.144828,0.22,1.519048
Automotive,18,8.0,0.041379,0.16,3.866667
Consumer / Retail / Food,53,6.0,0.121839,0.12,0.984906
Industrials / Aerospace / Defense,36,4.0,0.082759,0.08,0.966667
Media / Entertainment,18,3.0,0.041379,0.06,1.450000
Energy,38,2.0,0.087356,0.04,0.457895
Healthcare,33,2.0,0.075862,0.04,0.527273
Agriculture,2,0.0,0.004598,0.00,0.000000


In [138]:
from scipy.stats import kruskal

metric = "weighted_degree"

groups = [
    group[metric].dropna().values
    for _, group in centrality_df.groupby("industry_group")
    if len(group) >= 5
]

stat, p = kruskal(*groups)

print("Metric:", metric)
print("Kruskal-Wallis statistic:", stat)
print("p-value:", p)

Metric: weighted_degree
Kruskal-Wallis statistic: 27.654017562466226
p-value: 0.010109889992686975
